# Differential atmospheric parameters with q2

q2 uses MOOG and Fe I/Fe II excitation and ionization balance to estimate Teff, logg, model [Fe/H], and microturbulence relative to the Sun. Export measured EWs and atomic data to lines_q2.csv and initial parameters to stars_q2.csv. The q2 metallicity sets the subsequent model atmosphere; final plotted Fe abundances are measured by the element-abundance workflow.

## Workflow

Prepare inputs; detect and filter lines; review Fe Gaussian fits; apply manual rejections and local EW retries; inspect Fe abundance trends; solve differential parameters with q2; save the atmosphere dump and CSV.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import logging
import multiprocessing
from multiprocessing import Pool
import matplotlib.pyplot as plt
from scipy.stats import norm
import shutil
import q2

from inlist import target, PIPELINE_OPTIONS_OVERRIDE
from run_context import apply_context_globals
from atmos_lbl_pipeline import (
    step_log,
    load_or_prepare_inputs,
    run_find_linemasks_part1,
    filter_and_save_linemasks,
    plot_gaussian_fits_by_element,
    prepare_manual_line_edit_tables,
    apply_deleted_lines_to_atomic_linelist,
    fast_apply_lineqa_to_linemasks,
    set_line_ew_in_linemask,
    compute_fe_abundance,
    plot_fe_abundance_trends,
    run_q2_fit_and_export_inputs,
    save_q2_dump,
    export_q2_dump_csv,
)

ctx = apply_context_globals(
    globals(),
    target=target,
    pipeline_options_override=PIPELINE_OPTIONS_OVERRIDE,
    prepare_io=True,
)

# Synchronize Part I with PIPELINE_OPTIONS['run_find_linemasks'].
RUN_FIND_LINEMASKS = PIPELINE_OPTIONS.get("run_find_linemasks", True)

# Synchronize external EW mode with PIPELINE_OPTIONS['input_has_external_ews'].
USE_EXTERNAL_EW = PIPELINE_OPTIONS.get("input_has_external_ews", False)
_bedell_q2_tar_ew = None
_bedell_q2_sun_ew = None

# Skip spectral convolution, line detection, and plots in external EW mode.
RUN_PART1_SPECTRUM = RUN_FIND_LINEMASKS and (not USE_EXTERNAL_EW)


## Part I: Fe-line quality review

Configure inlist.PIPELINE_OPTIONS_OVERRIDE. Set run_find_linemasks=False to reuse existing fitted linemasks for a parameter rerun. External EW mode requires an existing template: pipeline-only Fe is the default, while non-Fe external rows can replace or extend the template. Outputs for external EW mode go to output/withEW_Bedell/<target>/.

In [ ]:
step_log("1", "Prepare spectrum and linelist inputs.")
_inputs = load_or_prepare_inputs(
    ispec=ispec,
    target=target,
    output_folder=output_folder,
    output_linelist_path=output_linelist_path,
    linelist_target_path=linelist_target_path,
    spectrum_norm_path=spectrum_norm_path,
    ispec_dir=ispec_dir,
    row_target=row_target,
    PIPELINE_OPTIONS=PIPELINE_OPTIONS,
    RUN_PART1_SPECTRUM=RUN_PART1_SPECTRUM,
    run_find_linemasks=RUN_FIND_LINEMASKS,
)

star_spectrum = _inputs["star_spectrum"]
_cres = _inputs.get("continuum_rescale_report")
if _cres:
    print("[continuum_rescale]", {k: _cres.get(k) for k in ("enabled","peak","offset","applied","scale","saved_path","peak_after")})
atomic_linelist = _inputs["atomic_linelist"]
atomic_linelist_file = _inputs["atomic_linelist_file"]
star_linemasks = _inputs["star_linemasks"]
iron_star_linemasks = _inputs["iron_star_linemasks"]
recover_star_linemasks = _inputs["recover_star_linemasks"]
linemasks = _inputs["linemasks"]
_bedell_q2_tar_ew = _inputs["bedell_q2_tar_ew"]
_bedell_q2_sun_ew = _inputs["bedell_q2_sun_ew"]


In [ ]:
if RUN_PART1_SPECTRUM:
    star_linemasks = run_find_linemasks_part1(
        ispec=ispec,
        star_spectrum=star_spectrum,
        atomic_linelist=atomic_linelist,
        from_resolution=from_resolution,
        to_resolution=to_resolution,
        min_depth=0.05,
        max_depth=1.00,
    )

In [ ]:
if RUN_PART1_SPECTRUM:
    _saved = filter_and_save_linemasks(
        ispec=ispec,
        target=target,
        output_folder=output_folder,
        star_linemasks=star_linemasks,
        ew_min=10,
        ew_max=100,
    )
    star_linemasks = _saved["star_linemasks"]
    iron_star_linemasks = _saved["iron_star_linemasks"]
    recover_star_linemasks = _saved["recover_star_linemasks"]
    linemask_output_folder = _saved["linemask_output_folder"]



- make the plot

In [ ]:
if RUN_PART1_SPECTRUM:
    _plot_stats = plot_gaussian_fits_by_element(
        star_spectrum=star_spectrum,
        linemasks=star_linemasks,
        output_dir=output_folder + "/figs_Fe_GaussianFits",
        element_filter=["Fe 1", "Fe 2"],
        prefix="FeFit",
        w_range=0.25,
        progress_every=10,
    )

### Prepare manual line-review tables

In [ ]:
if RUN_PART1_SPECTRUM:
    _qa_paths = prepare_manual_line_edit_tables(
        ispec=ispec, star_spectrum=star_spectrum, star_linemasks=star_linemasks,
        output_folder=output_folder,
        target=target,
        output_linelist_path=output_linelist_path,
        PIPELINE_OPTIONS=PIPELINE_OPTIONS,
    )
    deleted_file_path = _qa_paths["deleted_file_path"]
    modified_file_path = _qa_paths["modified_file_path"]
    step_log(
        "5",
        f"Manual edit tables ready: deleted={deleted_file_path}, modified={modified_file_path}",
    )

### Review and reject unsuitable line fits

Save the QA page before continuing. Local retries offer original and new EW choices; rejected second-pass lines remain excluded.

In [ ]:
if RUN_PART1_SPECTRUM:
    filtered_linelist = apply_deleted_lines_to_atomic_linelist(
        ispec=ispec,
        output_folder=output_folder,
        linelist_target_path=linelist_target_path,
    )
    atomic_linelist_file = linelist_target_path

### Rebuild and apply line decisions

In [ ]:
if RUN_PART1_SPECTRUM:
    _fast_lq = fast_apply_lineqa_to_linemasks(
        star_spectrum=star_spectrum,
        ispec=ispec,
        target=target,
        output_folder=output_folder,
        star_linemasks=star_linemasks,
        ew_min=10,
        ew_max=100,
    )
    star_linemasks = _fast_lq["star_linemasks"]
    iron_star_linemasks = _fast_lq["iron_star_linemasks"]
    recover_star_linemasks = _fast_lq["recover_star_linemasks"]
    step_log("7", f"Fast LineQA kept lines: {len(star_linemasks)}")


In [ ]:
step_log("7", "Function set_line_ew_in_linemask is imported from atmos_lbl_pipeline.py")


In [ ]:
lm_path = linemask_output_folder + f"/{target}_melendez2014_star_fitted_linemasks.txt"
df_ew_modify = pd.read_csv(output_folder+"linemasks/Lines_EW_modified.tsv", sep="\t")

use_A  = "wave_A"  in df_ew_modify.columns
use_nm = "wave_nm" in df_ew_modify.columns
if not (use_A or use_nm):
    raise ValueError("df_ew_modify requires wave_A or wave_nm.")

n_ok, n_err = 0, 0
for k, r in df_ew_modify.iterrows():
    try:
        elem = str(r["element"]).strip()
        ew   = float(r["ew_mA_new"])
        if use_A:
            lam  = float(r["wave_A"])
            unit = "A"
        else:
            lam  = float(r["wave_nm"])
            unit = "nm"

        # Match at the stored wavelength precision; adjust tol if needed.
        set_line_ew_in_linemask(lm_path, element=elem, wave_value=lam,
            ew_mA=ew, wave_unit=unit, tol=0.0, note_tag="EW_man=%.1f mA")
        n_ok += 1
    except Exception as e:
        print(f"[Skipped] row {k}: {r.get('element','?')} λ={r.get('wave_A', r.get('wave_nm','?'))} -> {e}")
        n_err += 1

print(f"Completed: successful {n_ok}; failed {n_err} lines.")

## Part II: Fe abundance diagnostics

Use MOOG, MARCS.GES, and Grevesse (2007). Inspect Fe I/Fe II means, line counts, scatter, and abundance trends with excitation potential and reduced EW. Review individual lines before attributing scatter to outliers.

In [ ]:
linemasks = ispec.read_line_regions(output_folder + f"/linemasks/{target}_melendez2014_star_fitted_linemasks.txt")
_fe_abund = compute_fe_abundance(
    ispec=ispec,
    linemasks=linemasks,
    ispec_dir=ispec_dir,
    initial_teff=initial_teff,
    initial_logg=initial_logg,
    initial_MH=initial_MH,
    initial_feh_err=row_target["[Fe/H]_err"],
)

linemasks = _fe_abund["linemasks"]
spec_abund = _fe_abund["spec_abund"]
normal_abund = _fe_abund["normal_abund"]
x_over_h = _fe_abund["x_over_h"]
x_over_fe = _fe_abund["x_over_fe"]
fe1_abund = _fe_abund["fe1_abund"]
fe2_abund = _fe_abund["fe2_abund"]
spec_abund_fe1 = _fe_abund["spec_abund_fe"]

model = _fe_abund["model"]
code = _fe_abund["code"]
teff = _fe_abund["teff"]
logg = _fe_abund["logg"]
MH = _fe_abund["MH"]
alpha = _fe_abund["alpha"]
microturbulence_vel = _fe_abund["microturbulence_vel"]
solar_abundances_file = _fe_abund["solar_abundances_file"]

initial_vmic = microturbulence_vel

In [ ]:
# Find indices with the requested Fe II abundance.
target_abund = -0.125

fe2_mask = (linemasks['element'] == "Fe 1") & (~np.isnan(x_over_h))
fe2_indices = np.where(fe2_mask)[0]
target_idx = fe2_indices[np.isclose(x_over_h[fe2_mask], target_abund, atol=1e-4)]

# Print the corresponding line information.
if len(target_idx) > 0:
    print(linemasks[target_idx][['wave_nm', 'element', 'ew', 'loggf']])
else:
    print("Didn't find the corresponding line.")

In [ ]:
plot_fe_abundance_trends(
    linemasks=linemasks,
    x_over_h=x_over_h,
    use_external_ew=USE_EXTERNAL_EW,
)

## Part III: Atmospheric-parameter fit

In [ ]:
# Read the spectrum and linelist.
#lineregions_path = linemask_output_folder+"/"+target+"_melendez2014_star_fitted_linemasks.txt"
# lineregions_ = star_linemasks

#line_regions = ispec.read_line_regions(ispec_dir + lineregions_path.format(code="width"))
#print(f"Successfully read {len(line_regions)} lines from the line regions.")

#--- Read the Normalized spectrum -------------------------------------------------------------
spectrum_path = spectrum_norm_path
normalized_star_spectrum = ispec.read_spectrum(spectrum_path)

from abund_plot import *
initial_vmic = ispec.estimate_vmic(initial_teff, initial_logg, initial_MH)

In [ ]:
_q2 = run_q2_fit_and_export_inputs(
    ispec=ispec,
    linemasks=linemasks,
    target=target,
    output_folder=output_folder,
    ispec_dir=ispec_dir,
    solar_label=solar_label,
    solar_lines_csv=solar_lines_csv,
    instrument_name=instrument_name,
    from_resolution=from_resolution,
    initial_teff=initial_teff,
    initial_logg=initial_logg,
    initial_MH=initial_MH,
    initial_vmic=initial_vmic,
    PIPELINE_OPTIONS=PIPELINE_OPTIONS,
    bedell_q2_tar_ew=globals().get("_bedell_q2_tar_ew"),
    bedell_q2_sun_ew=globals().get("_bedell_q2_sun_ew"),
)
linemasks_fe = _q2["linemasks_fe"]
q2_dir = _q2["q2_dir"]
solution_csv = _q2["solution_csv"]
row = _q2["row"]
teff_q2 = _q2["teff_q2"]
logg_q2 = _q2["logg_q2"]
feh_q2 = _q2["feh_q2"]
vt_q2 = _q2["vt_q2"]
# use step_log to print the reference parameters
step_log("10", f"Reference parameters: Teff = {initial_teff}, logg = {initial_logg}, [Fe/H] = {initial_MH}")


# Saving

In [ ]:
_dump = save_q2_dump(
    ispec=ispec,
    output_atmos_params_dumpfile_path=output_atmos_params_dumpfile_path,
    solution_row=row,
    linemasks_fe=linemasks_fe,
    target=target,
    solution_csv=solution_csv,
)
dump_file_q2 = _dump["dump_file_q2"]
params_q2 = _dump["params_q2"]
errors_q2 = _dump["errors_q2"]

csv_atmos_q2 = export_q2_dump_csv(
    ispec=ispec,
    target=target,
    output_atmos_params_dumpfile_path=output_atmos_params_dumpfile_path,
)
